# Data Generation for Telescope image finding using CNNs

The pipeline works as follows:
- using RA and Dec coordinates to retrieve 60'x 60' finder images SkyView 
- put any generated images in the *finder_images* folder
- create the telescope image by croping the finder image, fliping the image horizontally, increasing contrast, adding gaussian noise
- add each of these images to the *telescope_images* folder
- write the path of the finder image, telescope image, and area of the crop to a CSV file 

# Generate finder charts 

In [1]:
import os
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from astropy import units as u
from astroquery.skyview import SkyView
import matplotlib.pyplot as plt

#the following block of code was written with the help of chatGPT

# --------------------------------------------------
# 1. Output folder
# --------------------------------------------------
dirname = "finder_images"
os.makedirs(dirname, exist_ok=True)

# --------------------------------------------------
# 2. Coordinates (example list)   this section I used ChatGPT to generate a python file containing X valid sky coordinates
# --------------------------------------------------
coords = [
    SkyCoord("14h25m27s", "-77d42m53s"),  # Obj1900
    SkyCoord("21h37m17s", "+19d42m24s"),  # Obj1901
    SkyCoord("18h19m36s", "+84d03m07s"),  # Obj1902
    SkyCoord("19h44m30s", "-67d10m14s"),  # Obj1903
    SkyCoord("23h11m54s", "+78d10m16s"),  # Obj1904
    SkyCoord("19h45m29s", "+08d03m08s"),  # Obj1905
    SkyCoord("19h38m43s", "-79d27m04s"),  # Obj1906
    SkyCoord("04h58m12s", "+35d12m01s"),  # Obj1907
    SkyCoord("23h29m27s", "+39d29m13s"),  # Obj1908
    SkyCoord("01h05m10s", "-09d06m02s"),  # Obj1909
    SkyCoord("18h46m34s", "+81d51m21s"),  # Obj1910
    SkyCoord("22h09m17s", "-69d46m42s"),  # Obj1911
    SkyCoord("06h39m55s", "+85d49m39s"),  # Obj1912
    SkyCoord("05h04m30s", "-29d22m44s"),  # Obj1913
    SkyCoord("20h43m15s", "+66d03m53s"),  # Obj1914
    SkyCoord("10h50m07s", "-49d39m47s"),  # Obj1915
    SkyCoord("03h29m11s", "+33d21m48s"),  # Obj1916
    SkyCoord("23h30m44s", "+40d50m16s"),  # Obj1917
    SkyCoord("18h27m06s", "+66d27m25s"),  # Obj1918
    SkyCoord("20h16m23s", "-19d47m35s"),  # Obj1919
    SkyCoord("13h46m27s", "+67d53m23s"),  # Obj1920
    SkyCoord("04h08m24s", "+52d00m07s"),  # Obj1921
    SkyCoord("11h17m42s", "-25d50m21s"),  # Obj1922
    SkyCoord("14h20m14s", "+44d00m08s"),  # Obj1923
    SkyCoord("23h00m55s", "+23d11m34s"),  # Obj1924
    SkyCoord("22h03m47s", "-70d41m43s"),  # Obj1925
    SkyCoord("03h30m31s", "-68d57m36s"),  # Obj1926
    SkyCoord("23h36m39s", "+60d35m54s"),  # Obj1927
    SkyCoord("20h14m24s", "-61d19m23s"),  # Obj1928
    SkyCoord("19h09m54s", "+66d59m05s"),  # Obj1929
    SkyCoord("17h37m28s", "-24d53m18s"),  # Obj1930
    SkyCoord("06h40m03s", "-83d15m32s"),  # Obj1931
    SkyCoord("15h25m05s", "-45d26m43s"),  # Obj1932
    SkyCoord("11h13m38s", "-47d49m45s"),  # Obj1933
    SkyCoord("12h54m32s", "-87d12m00s"),  # Obj1934
    SkyCoord("07h11m59s", "+07d04m33s"),  # Obj1935
    SkyCoord("13h03m54s", "+68d36m58s"),  # Obj1936
    SkyCoord("09h32m51s", "+33d22m18s"),  # Obj1937
    SkyCoord("15h01m57s", "+68d53m45s"),  # Obj1938
    SkyCoord("00h10m35s", "+73d59m56s"),  # Obj1939
    SkyCoord("02h51m18s", "+61d45m26s"),  # Obj1940
    SkyCoord("17h34m18s", "+53d41m14s"),  # Obj1941
    SkyCoord("04h49m33s", "+01d51m27s"),  # Obj1942
    SkyCoord("23h06m37s", "+61d24m52s"),  # Obj1943
    SkyCoord("11h18m21s", "+84d08m35s"),  # Obj1944
    SkyCoord("08h24m39s", "-61d16m17s"),  # Obj1945
    SkyCoord("14h18m42s", "-55d53m28s"),  # Obj1946
    SkyCoord("23h14m13s", "-68d32m37s"),  # Obj1947
    SkyCoord("00h35m21s", "-65d19m32s"),  # Obj1948
    SkyCoord("04h07m11s", "+40d43m19s"),  # Obj1949
    SkyCoord("10h17m46s", "-17d21m45s"),  # Obj1950
    SkyCoord("05h41m03s", "-37d29m20s"),  # Obj1951
    SkyCoord("09h49m35s", "+66d40m21s"),  # Obj1952
    SkyCoord("10h11m46s", "+66d11m28s"),  # Obj1953
    SkyCoord("06h12m39s", "-43d39m03s"),  # Obj1954
    SkyCoord("06h10m10s", "+79d59m31s"),  # Obj1955
    SkyCoord("21h12m47s", "+07d25m51s"),  # Obj1956
    SkyCoord("12h09m49s", "+77d06m30s"),  # Obj1957
    SkyCoord("23h01m14s", "-89d09m42s"),  # Obj1958
    SkyCoord("04h08m18s", "+87d11m03s"),  # Obj1959
    SkyCoord("09h07m50s", "+56d41m44s"),  # Obj1960
    SkyCoord("07h37m36s", "-27d29m16s"),  # Obj1961
    SkyCoord("13h41m06s", "+26d14m29s"),  # Obj1962
    SkyCoord("20h06m39s", "+54d07m38s"),  # Obj1963
    SkyCoord("04h42m07s", "-66d26m05s"),  # Obj1964
    SkyCoord("10h48m47s", "-32d08m20s"),  # Obj1965
    SkyCoord("04h59m50s", "+57d24m10s"),  # Obj1966
    SkyCoord("01h28m22s", "-01d23m37s"),  # Obj1967
    SkyCoord("20h19m08s", "+72d31m06s"),  # Obj1968
    SkyCoord("11h46m35s", "-12d50m54s"),  # Obj1969
    SkyCoord("22h48m31s", "-25d10m33s"),  # Obj1970
    SkyCoord("23h49m15s", "+38d16m53s"),  # Obj1971
    SkyCoord("20h25m08s", "-83d04m27s"),  # Obj1972
    SkyCoord("09h12m31s", "+42d22m15s"),  # Obj1973
    SkyCoord("05h57m23s", "+30d21m52s"),  # Obj1974
    SkyCoord("13h29m39s", "-20d08m06s"),  # Obj1975
    SkyCoord("07h19m13s", "-77d52m19s"),  # Obj1976
    SkyCoord("17h29m58s", "-75d24m19s"),  # Obj1977
    SkyCoord("05h58m47s", "-75d54m58s"),  # Obj1978
    SkyCoord("23h23m49s", "-06d51m36s"),  # Obj1979
    SkyCoord("22h33m44s", "-10d06m47s"),  # Obj1980
    SkyCoord("01h29m32s", "-61d16m51s"),  # Obj1981
    SkyCoord("03h06m38s", "+18d28m58s"),  # Obj1982
    SkyCoord("19h08m25s", "-32d43m54s"),  # Obj1983
    SkyCoord("06h12m40s", "+19d57m53s"),  # Obj1984
    SkyCoord("20h51m27s", "-11d54m59s"),  # Obj1985
    SkyCoord("16h04m05s", "-78d02m04s"),  # Obj1986
    SkyCoord("19h53m24s", "-65d51m12s"),  # Obj1987
    SkyCoord("11h38m08s", "-02d53m49s"),  # Obj1988
    SkyCoord("07h58m43s", "-13d37m52s"),  # Obj1989
    SkyCoord("11h20m44s", "+29d03m50s"),  # Obj1990
    SkyCoord("00h48m06s", "+74d45m59s"),  # Obj1991
    SkyCoord("22h27m48s", "+28d17m08s"),  # Obj1992
    SkyCoord("09h39m42s", "-00d18m00s"),  # Obj1993
    SkyCoord("00h33m31s", "-25d37m25s"),  # Obj1994
    SkyCoord("19h35m08s", "-21d55m51s"),  # Obj1995
    SkyCoord("00h15m10s", "+47d23m39s"),  # Obj1996
    SkyCoord("00h07m38s", "-08d31m46s"),  # Obj1997
    SkyCoord("19h05m48s", "-18d00m25s"),  # Obj1998
    SkyCoord("13h25m47s", "+11d18m53s"),  # Obj1999
    SkyCoord("22h36m31s", "+89d41m02s"),  # Obj2000
]

# --------------------------------------------------
# 3. Function to generate an DSS finder chart
# --------------------------------------------------
def make_dss_finder(coord, name, fov_arcmin=60):
    radius = (fov_arcmin / 2) * u.arcmin
    images = SkyView.get_images(
        position=coord,
        survey=["DSS2 Red"],
        radius=radius,
        pixels=1200  # Try for large field, SkyView may limit resolution
    )

    hdu = images[0][0]
    wcs = WCS(hdu.header)

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": wcs})
    ax.imshow(hdu.data, cmap='gray', origin='lower')

    ax.axis('off')

    return fig

# --------------------------------------------------
# 4. Loop through targets
# --------------------------------------------------
for i, coord in enumerate(coords, start=1):
    name = f"target_{i:04d}"
    png_path = os.path.join(dirname, f"{name}.png")

    fig = make_dss_finder(coord, name, fov_arcmin=60)
    fig.savefig(png_path, dpi=150, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    if i == 1 or i % 10 == 0:
        print(f"Generated DSS finder: {png_path}")

#end of where chatGPT helped 


Generated DSS finder: finder_images\target_0001.png
Generated DSS finder: finder_images\target_0010.png
Generated DSS finder: finder_images\target_0020.png
Generated DSS finder: finder_images\target_0030.png
Generated DSS finder: finder_images\target_0040.png
Generated DSS finder: finder_images\target_0050.png
Generated DSS finder: finder_images\target_0060.png
Generated DSS finder: finder_images\target_0070.png
Generated DSS finder: finder_images\target_0080.png
Generated DSS finder: finder_images\target_0090.png
Generated DSS finder: finder_images\target_0100.png


# Genrate telescope images 

In [6]:
from PIL import Image, ImageEnhance
import random 

import numpy as np
 

def make_telescope_img(input_path):
    try:
        img = Image.open(input_path) 
    except FileNotFoundError:
        print("Error: Image file not found. Please check the path.")
        exit()

        
    # --------------------------------------------------
    #crop image
    # --------------------------------------------------
    width, height = img.size
    t_width = round(0.25*width)
    t_height = round(0.25*height)
    
    x_coord = random.randint(0, width - t_width)
    y_coord = random.randint(0, height - t_height)

    crop_area = (x_coord, y_coord, x_coord + t_width, y_coord + t_height)
    cropped_img = img.crop(crop_area)

    
    # --------------------------------------------------
    #flip image 
    # --------------------------------------------------
    flipped_img = cropped_img.transpose(Image.FLIP_LEFT_RIGHT)

    
    # --------------------------------------------------
    #increases contrast of image
    # --------------------------------------------------
    enhancer = ImageEnhance.Contrast(flipped_img)
    enhanced_image = enhancer.enhance(random.uniform(1,2))

    
    # --------------------------------------------------
    #add noise to image 
    # --------------------------------------------------
    noise_image = np.array(enhanced_image.convert("RGB"), dtype=np.float32)
    # Generate 2 dimensional Gaussian noise 
    noise = np.random.normal(loc=0.0, scale=10, size=noise_image.shape[:2])
    #stack noise to return to rgb 
    noise = np.stack((noise,noise,noise), axis = 2)
    # Add noise + clip to [0,255]
    noisy = np.clip(noise_image + noise, 0, 255).astype(np.uint8)
    telescope_img = Image.fromarray(noisy)


    final_width = img.width
    final_height = img.height + telescope_img.height

    if(final_width<final_height):
        final_width = final_height

    # black background
    combined = Image.new("RGB", (final_width, final_height), color=(0, 0, 0))

    # paste finder image
    combined.paste(img, (0, 0))



    combined.paste(telescope_img, (0, img.height))

    return combined, crop_area

# Write data to CSV

In [ ]:
import csv 
import os

# --------------------------------------------------
# 1. Output folder
# --------------------------------------------------
dirname = "telescope_images"
os.makedirs(dirname, exist_ok=True)

output_csv = os.path.join(dirname, "data.csv")
all_data = []   # list to store metadata for each image


counter_dict = {}

# --------------------------------------------------
# loop through finder images 
# --------------------------------------------------
for filename in os.listdir("finder_images"):

   
    if filename.endswith(".png"):

        
        base_name = filename[:-4]
        full_path = os.path.join("finder_images", filename)
        print(f"Processing PNG file: {full_path}")

        if base_name not in counter_dict:
            counter_dict[base_name] = 1

        for i in range(10):
            # Generate telescope image
            telescope_img, crop_area = make_telescope_img(full_path)
    
            # Create file name: (findername)_telescope_####
            count = counter_dict[base_name]
            out_name = f"{base_name}_telescope_{count:04d}.png"
            out_path = os.path.join(dirname, out_name)
    
            # Save the telescope image
            telescope_img.save(out_path)
            print(f"Saved: {out_path}")
    
            # Increment counter for that finder image
            counter_dict[base_name] += 1
    
            #CSV data for telescope image 
            data = {
                "filename": out_path,
                "xmin": crop_area[0],
                "ymin": crop_area[1],
                "xmax": crop_area[2],
                "ymax": crop_area[3]
            }
            all_data.append(data)


# --------------------------------------------------
# put data in CSV file 
# --------------------------------------------------
with open(output_csv, "w", newline="") as f:
    if len(all_data) == 0:
        print("No data to write.")
    else:
        writer = csv.DictWriter(f, fieldnames=all_data[0].keys())
        writer.writeheader()
        writer.writerows(all_data)

print(f"CSV saved to {output_csv}")

Processing PNG file: finder_images\target_0001.png
Saved: telescope_images\target_0001_telescope_0001.png
Saved: telescope_images\target_0001_telescope_0002.png
Saved: telescope_images\target_0001_telescope_0003.png
Saved: telescope_images\target_0001_telescope_0004.png
Saved: telescope_images\target_0001_telescope_0005.png
Saved: telescope_images\target_0001_telescope_0006.png
Saved: telescope_images\target_0001_telescope_0007.png
Saved: telescope_images\target_0001_telescope_0008.png
Saved: telescope_images\target_0001_telescope_0009.png
Saved: telescope_images\target_0001_telescope_0010.png
Processing PNG file: finder_images\target_0002.png
Saved: telescope_images\target_0002_telescope_0001.png
Saved: telescope_images\target_0002_telescope_0002.png
Saved: telescope_images\target_0002_telescope_0003.png
Saved: telescope_images\target_0002_telescope_0004.png
Saved: telescope_images\target_0002_telescope_0005.png
Saved: telescope_images\target_0002_telescope_0006.png
Saved: telescope_i